# SMS Spam Detection using NLP
**Course:** DS483 – Natural Language Processing  
**Dataset:** UCI SMS Spam Collection (5,574 labeled messages)

This notebook implements all five project tasks: preprocessing, TF-IDF/cosine similarity, Naive Bayes and Logistic Regression, LSTM, and fine-tuning a BERT-family model.

> Run from top to bottom in Google Colab. For the BERT section, select **Runtime → Change runtime type → T4 GPU** if available. All reported metrics are generated from the actual run; do not copy example or assumed scores into the report.

## 0. Environment setup

In [ ]:
import re, html, os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings("ignore")
nltk.download("stopwords")


## 1. Load and explore the UCI dataset

In [ ]:
# Load dataset
df = pd.read_csv('/kaggle/input/datasets/ossamasamir/smsspamcollection/SMSSpamCollection',sep='\t',header=None,names=['label', 'message'])


In [ ]:
df.head()

In [ ]:
# Separate features and target
X_raw = df[['message']]
y_raw = df['label']

In [ ]:
# Display information
print("Feature columns:", X_raw.columns.tolist())
print("Target columns:", y_raw.name)

display(df.head())

print("\nDataset Shape:", df.shape)
print("\nClass Distribution:")
print(y_raw.value_counts())

In [ ]:
df["label"] = df["label"].replace({"ham": 0, "spam": 1})
df = df.dropna().drop_duplicates().reset_index(drop=True)
df = df[df["label"].isin([0, 1])].copy()
df["label"] = df["label"].astype(int)

print("Dataset shape:", df.shape)
print("Missing values:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print(df["label"].value_counts().rename(index={0:"ham", 1:"spam"}))


In [ ]:
df.head()

In [ ]:
# Class distribution
counts = df["label"].map({0:"Ham", 1:"Spam"}).value_counts()
plt.figure(figsize=(6,4))
sns.barplot(x=counts.index, y=counts.values)
plt.title("SMS class distribution")
plt.xlabel("Class"); plt.ylabel("Number of messages")
plt.show()

print("Class percentages:")
display((counts / counts.sum() * 100).round(2).rename("percentage").to_frame())


### Explore textual characteristics
Spam messages often contain promotional language, urgency, prizes, money-related terms, phone numbers, URLs, and unusual punctuation. These are tendencies, not strict rules.

In [ ]:
df["char_count"] = df["message"].str.len()
df["word_count"] = df["message"].str.split().str.len()
df["digit_count"] = df["message"].str.count(r"\\d")
df["url_flag"] = df["message"].str.contains(r"http|www\\.", case=False, regex=True)
df["exclamation_count"] = df["message"].str.count("!")

display(df.groupby("label")[["char_count","word_count","digit_count",
                             "url_flag","exclamation_count"]].mean().round(2)
        .rename(index={0:"Ham",1:"Spam"}))

fig, axes = plt.subplots(1,2,figsize=(12,4))
sns.boxplot(data=df, x="label", y="word_count", ax=axes[0])
axes[0].set_xticklabels(["Ham","Spam"]); axes[0].set_title("Words per message")
sns.boxplot(data=df, x="label", y="char_count", ax=axes[1])
axes[1].set_xticklabels(["Ham","Spam"]); axes[1].set_title("Characters per message")
plt.tight_layout(); plt.show()

display(df.sample(min(10, len(df)), random_state=SEED)[["message","label"]]
        .replace({"label":{0:"ham",1:"spam"}}))


## 2. Task 1 – Text preprocessing

The classical machine-learning pipeline uses the following steps:

1. Convert text to lowercase.
2. Decode HTML entities.
3. Remove URLs and email addresses.
4. Replace digit sequences with a marker (`number`) rather than silently discarding all numerical information.
5. Remove punctuation and non-letter symbols.
6. Tokenize into words.
7. Remove English stopwords while preserving negations such as **not**, **no**, and **nor**.
8. Apply Porter stemming to reduce related word forms.

Negation words are deliberately retained because removing them can reverse the meaning of a message. The original messages are preserved for the LSTM/BERT models, which use their own tokenization and can benefit from word order and context.

In [ ]:
nltk.download("stopwords")
stop_words = set(stopwords.words("english")) - {"no", "nor", "not"}
stemmer = PorterStemmer()

def preprocess_text(text):
    text = html.unescape(str(text)).lower()
    text = re.sub(r"https?://\\S+|www\\.\\S+", " url ", text)
    text = re.sub(r"\\b[\\w.+-]+@[\\w.-]+\\.[a-z]{2,}\\b", " email ", text)
    text = re.sub(r"\\d+", " number ", text)
    text = re.sub(r"[^a-z_\\s]", " ", text)
    tokens = text.split()
    tokens = [stemmer.stem(w) for w in tokens
              if w not in stop_words and len(w) > 1]
    return " ".join(tokens)

df["clean_message"] = df["message"].apply(preprocess_text)
for _, row in df.sample(5, random_state=SEED).iterrows():
    print("BEFORE:", row["message"])
    print("AFTER: ", row["clean_message"])
    print("-" * 90)


## 3. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_message"], df["label"],
    test_size=0.20, random_state=SEED, stratify=df["label"]
)
print("Train:", len(X_train), "Test:", len(X_test))
print("Train class ratio:\n", y_train.value_counts(normalize=True))
print("Test class ratio:\n", y_test.value_counts(normalize=True))


## 4. Task 2 – TF-IDF and cosine similarity

TF-IDF represents each message as a sparse numerical vector. Term Frequency reflects how often a term appears in a message, while Inverse Document Frequency reduces the weight of terms that occur in many messages. Cosine similarity measures the angle between two vectors; values closer to 1 indicate more similar term-weight patterns.

We use **unigrams and bigrams** so phrases such as “free entry” can be represented. The vectorizer is fitted on training data only to avoid test-set leakage.

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.98,
                         sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF train matrix:", X_train_tfidf.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))


In [ ]:
# Retrieve the most similar test messages for a selected query
similarity_matrix = cosine_similarity(X_test_tfidf)
test_messages = X_test.reset_index(drop=True)

query_index = 0
scores = similarity_matrix[query_index].copy()
scores[query_index] = -1  # exclude the query itself
top_indices = np.argsort(scores)[::-1][:5]

print("QUERY MESSAGE:")
print(test_messages.iloc[query_index])
print("\\nTOP SIMILAR MESSAGES:")
for idx in top_indices:
    print(f"Similarity: {scores[idx]:.3f} | {test_messages.iloc[idx]}")


### Similarity interpretation

A high cosine score means two messages share important terms or phrases after TF-IDF weighting. It does not guarantee that they have the same label: two messages can discuss the same topic but have different intent. TF-IDF is also mainly lexical and does not fully understand paraphrases or context.

## 5. Task 3 – Naive Bayes and Logistic Regression

Both models use the same training-only TF-IDF representation for a fair comparison. Multinomial Naive Bayes estimates class probabilities from word counts/weights using Bayes' rule and a smoothing parameter. Logistic Regression is a discriminative linear classifier that estimates class probabilities and learns feature weights with regularization.

In [ ]:
nb_model = MultinomialNB(alpha=0.5)
lr_model = LogisticRegression(max_iter=2000, class_weight=None,
                              random_state=SEED, C=3.0)

nb_model.fit(X_train_tfidf, y_train)
lr_model.fit(X_train_tfidf, y_train)

pred_nb = nb_model.predict(X_test_tfidf)
pred_lr = lr_model.predict(X_test_tfidf)

def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (spam)": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Recall (spam)": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1 (spam)": f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    }

classical_results = [
    evaluate_model("Multinomial Naive Bayes", y_test, pred_nb),
    evaluate_model("Logistic Regression", y_test, pred_lr)
]
classical_results_df = pd.DataFrame(classical_results).set_index("Model")
display(classical_results_df.round(4))

print("NAIVE BAYES REPORT\n", classification_report(y_test, pred_nb,
      target_names=["ham","spam"], zero_division=0))
print("LOGISTIC REGRESSION REPORT\n", classification_report(y_test, pred_lr,
      target_names=["ham","spam"], zero_division=0))


In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1,2,figsize=(10,4))
for ax, pred, title in zip(axes, [pred_nb,pred_lr],
                           ["Naive Bayes","Logistic Regression"]):
    cm = confusion_matrix(y_test, pred, labels=[0,1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Ham","Spam"], yticklabels=["Ham","Spam"], ax=ax)
    ax.set_title(title); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()


In [ ]:
# Incorrectly classified examples (false positives and false negatives)
def show_errors(name, y_true, y_pred, raw_test):
    err = pd.DataFrame({
        "message": raw_test.reset_index(drop=True),
        "actual": y_true.reset_index(drop=True),
        "predicted": pd.Series(y_pred)
    })
    err = err[err.actual != err.predicted].copy()
    err["error_type"] = np.where(
        (err.actual == 0) & (err.predicted == 1),
        "False positive: ham classified as spam",
        "False negative: spam classified as ham"
    )
    print(name, "- errors:", len(err))
    display(err.head(12).replace({"actual":{0:"ham",1:"spam"},
                                  "predicted":{0:"ham",1:"spam"}}))
show_errors("Naive Bayes", y_test, pred_nb, df.loc[X_test.index,"message"])
show_errors("Logistic Regression", y_test, pred_lr, df.loc[X_test.index,"message"])


## 6. Task 4 – LSTM sequence-based classifier

The LSTM receives integer token sequences rather than TF-IDF vectors. An Embedding layer learns dense word representations, an LSTM layer processes token order and context, and a sigmoid output predicts the spam probability. Padding makes sequences in a batch have equal length.

We use the original message text (lowercase only) rather than stemming and stopword removal, because sequence models may need grammatical context and the original word order.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models, callbacks

tf.keras.utils.set_random_seed(SEED)

# Split training data again to create a validation set
X_lstm_train, X_lstm_val, y_lstm_train, y_lstm_val = train_test_split(
    df.loc[X_train.index, "message"], y_train,
    test_size=0.15, random_state=SEED, stratify=y_train
)
X_lstm_test = df.loc[X_test.index, "message"]

MAX_WORDS = 12000
MAX_LEN = 80

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_lstm_train)

def encode_texts(texts):
    seq = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")

xtr_seq = encode_texts(X_lstm_train)
xval_seq = encode_texts(X_lstm_val)
xtest_seq = encode_texts(X_lstm_test)

lstm_model = models.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(MAX_WORDS, 64, mask_zero=True),
    layers.Bidirectional(layers.LSTM(64, dropout=0.2)),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])
lstm_model.compile(optimizer="adam", loss="binary_crossentropy",
                   metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
                            tf.keras.metrics.Recall(name="recall")])
lstm_model.summary()


In [ ]:
early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=2,
                                       restore_best_weights=True)
history = lstm_model.fit(
    xtr_seq, np.asarray(y_lstm_train),
    validation_data=(xval_seq, np.asarray(y_lstm_val)),
    epochs=10, batch_size=32, callbacks=[early_stop], verbose=1
)

lstm_prob = lstm_model.predict(xtest_seq).ravel()
lstm_pred = (lstm_prob >= 0.5).astype(int)
print(classification_report(y_test, lstm_pred,
      target_names=["ham","spam"], zero_division=0))


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("LSTM learning curves")
plt.legend(); plt.show()

cm = confusion_matrix(y_test, lstm_pred, labels=[0,1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Ham","Spam"], yticklabels=["Ham","Spam"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("LSTM confusion matrix")
plt.show()


## 7. Task 5 – Fine-tune a BERT-family model

This section fine-tunes **DistilBERT**, a smaller BERT-family transformer, for binary sequence classification. The pretrained tokenizer handles subword tokenization, and the model uses contextual self-attention representations. Unlike the classical pipeline, we do not remove stopwords, punctuation, or apply stemming before tokenization.

This section needs more compute and may take several minutes. It is suitable for a Colab GPU. The test set is held out and is used only for final evaluation.

In [ ]:
%pip install --force-reinstall "pyarrow==18.1.0" "huggingface-hub==0.27.1" "tokenizers==0.20.3" "transformers==4.46.3" "datasets==3.2.0"

In [ ]:
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

MODEL_NAME = "distilbert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Use original SMS text, split training data into train and validation
bert_train_df = pd.DataFrame({
    "text": df.loc[X_train.index, "message"].values,
    "label": y_train.values
})
bert_train_part, bert_val_part = train_test_split(
    bert_train_df, test_size=0.15, random_state=SEED,
    stratify=bert_train_df["label"]
)
bert_test_df = pd.DataFrame({
    "text": df.loc[X_test.index, "message"].values,
    "label": y_test.values
})

def make_hf_dataset(frame):
    ds = Dataset.from_pandas(frame.reset_index(drop=True))
    def tokenize(batch):
        return bert_tokenizer(batch["text"], truncation=True, max_length=128)
    return ds.map(tokenize, batched=True).remove_columns(["text"])

train_ds = make_hf_dataset(bert_train_part)
val_ds = make_hf_dataset(bert_val_part)
test_ds = make_hf_dataset(bert_test_df)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)
data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    p, r, f, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    return {"accuracy": accuracy_score(labels, preds),
            "precision_spam": p, "recall_spam": r, "f1_spam": f}


In [ ]:
training_args = TrainingArguments(
    output_dir="./sms-distilbert-results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_spam",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    processing_class=bert_tokenizer,
    compute_metrics=compute_metrics
)
trainer.train()


In [ ]:
bert_output = trainer.predict(test_ds)
bert_pred = np.argmax(bert_output.predictions, axis=1)
print(classification_report(y_test, bert_pred,
      target_names=["ham","spam"], zero_division=0))


## 8. Compare all models

In [ ]:
all_results = [
    evaluate_model("Multinomial Naive Bayes", y_test, pred_nb),
    evaluate_model("Logistic Regression", y_test, pred_lr),
    evaluate_model("BiLSTM", y_test, lstm_pred),
    evaluate_model("DistilBERT", y_test, bert_pred)
]
results_df = pd.DataFrame(all_results).set_index("Model")
display(results_df.sort_values("F1 (spam)", ascending=False).round(4))

results_df[["Precision (spam)","Recall (spam)","F1 (spam)"]].plot(
    kind="bar", figsize=(11,5), ylim=(0,1), rot=20
)
plt.title("Spam-class performance comparison")
plt.ylabel("Score"); plt.grid(axis="y", alpha=0.3); plt.tight_layout()
plt.show()


## 9. Error analysis across models

In [ ]:
predictions = {
    "Naive Bayes": pred_nb,
    "Logistic Regression": pred_lr,
    "BiLSTM": lstm_pred,
    "DistilBERT": bert_pred
}
raw_test = df.loc[X_test.index, "message"].reset_index(drop=True)
error_rows = []
for model_name, preds in predictions.items():
    for i, (actual, predicted) in enumerate(zip(y_test.reset_index(drop=True), preds)):
        if actual != predicted:
            error_rows.append({
                "model": model_name, "message": raw_test.iloc[i],
                "actual": "spam" if actual == 1 else "ham",
                "predicted": "spam" if predicted == 1 else "ham",
                "error": "false negative" if actual == 1 else "false positive"
            })
errors_df = pd.DataFrame(error_rows)
print("Total model-error records:", len(errors_df))
display(errors_df.head(25))


## 10. Deployment recommendation and conclusion

Use the measured results above to select the final model. Do not select solely by accuracy: spam detection is class-imbalanced, and **spam recall** measures how many actual spam messages are caught, while **spam precision** measures how many messages flagged as spam are truly spam. False positives are important because legitimate SMS messages should not be blocked.

A practical deployment decision should consider:
- Spam-class precision, recall, and F1-score.
- False positives and false negatives.
- Inference latency, memory, model size, and hardware availability.
- Ease of retraining and maintenance.

**Expected trade-off:** TF-IDF + Logistic Regression or Multinomial Naive Bayes is lightweight, fast, and easy to deploy. DistilBERT can model context more deeply but requires more memory and compute. LSTM models word order but add training and inference complexity. The best deployment choice is the model that achieves an acceptable spam recall with a low false-positive rate under the company's latency and hardware constraints.

### Suggested report conclusion (complete it using your measured table)
The experiments compared four approaches to SMS spam classification: Multinomial Naive Bayes, Logistic Regression, BiLSTM, and DistilBERT. All models were evaluated on the same stratified held-out test set using accuracy, spam precision, spam recall, and spam F1-score. The final model should be selected by balancing detection quality and operational complexity. For a resource-constrained SMS gateway, a strong TF-IDF linear classifier may be preferable because it is fast and interpretable. If the transformer produces a meaningful improvement in spam recall/F1 without an unacceptable increase in latency and infrastructure cost, DistilBERT may be justified.

### Reproducibility notes
- Fixed random seed: 42.
- Stratified 80/20 train/test split.
- TF-IDF vocabulary learned from training data only.
- LSTM/BERT validation data comes only from the training partition.
- The test partition is reserved for final evaluation.
- Run the notebook in one environment and report the actual results produced by that run.
